# Задача
Сделать простого чат-бота на LangGraph, который:

1. Общается в формате: User → LLM → User → LLM → ...

2. Умеет останавливаться на `interrupt` и ждать ответа пользователя

3. Запускается в стриминге

4. Имеет простой Gradio-интерфейс

Решение можно оформить в виде notebook либо в виде небольшого репозитория

## Условия
Что хочется, чтобы было реализовано:
1. Граф, в котором будем как минимум две ноды, одна из которых interrupt нода, и этот граф будет наследоваться от `BaseGraph`
3. Методы `build_graph`, `stream_start` и `stream_events` в классе `BaseGraph`
4. Простенький интерфейс (класс `Interface`) на Gradio с использованием `gr.ChatBot()`

## BaseGraph

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

class BaseGraph:
    """
    Base class for building LangGraph conversational graphs.
    """
    def __init__(self, model):
        self.model = model
        self.checkpointer = InMemorySaver()
        self.graph = self.build_graph()

    def build_graph(self):
        """
        Должен вернуть скомпилированный граф LangGraph.
        """
        raise NotImplementedError

    def stream_start(self, *args, **kwargs):
        """
        Запускает граф «с нуля»: отправляет пустой dict {}, чтобы сработал
        первый interrupt и вернулся вопрос про новую задачу.
        """
        raise NotImplementedError(
            "Реализуйте stream_start в вашем классе и продумайте, "
            "как запускать граф с нуля и что стримить наружу."
        )

    def stream_events(self, *args, **kwargs):
        """
        Стриминг ответа графа на сообщение пользователя.
        Должен обрабатывать пользовательский ввод и возвращать поток событий, по которому UI может восстановить ход диалога

        Шаги:
          - продумайте сигнатуру (user_message, thread_id, pending_interrupt и/или что-то еще),
          - сформируйте input для self.graph.stream(...),
          - обработайте interrupt'ы,
          - определите формат выходных событий, удобный для вашего UI.
        """
        raise NotImplementedError(
            "Реализуйте stream_events в вашем классе и продумайте логику стриминга."
        )

    # Используете этот метод для формирования config внутри `self.stream_events`.
    @staticmethod
    def create_config(thread_id: str):
        return {"configurable": {"thread_id": thread_id}}

    # Полезные методы для вывода состояния графа и его отрисовки
    def get_state(self, thread_id, subgraphs: bool = True):
        config = self.create_config(thread_id)
        snapshot = self.graph.get_state(config=config, subgraphs=subgraphs)
        return snapshot.values

    def draw_graph(self, xray: bool = True, **kwargs):
        from IPython.display import Image, display
        display(
            Image(self.graph.get_graph(xray=xray).draw_mermaid_png()),
            **kwargs,
        )

## Gradio интерфейс

In [9]:
import gradio as gr
import uuid

class Interface:
    def __init__(self, graph):
        self.graph = graph

    def on_load(self, *args, **kwargs):
        """Коллбек, вызываемый при загрузке интерфейса.
        Предполагается, что здесь вызывается метод `stream_start` вашего графа
        """
        ...

    def on_submit(self, *args, **kwargs):
        """Коллбек, вызываемый при отправке сообщения пользователем.
        Предполагается, что здесь вызывается метод `stream_events` вашего графа, который будет
        стримить в `chat_history` вашего Chatbot.
        """
        ...

    def create_interface(self):
        with gr.Blocks() as demo:

            # Базовый функционал для чатбота
            chatbot = gr.Chatbot(type="messages")
            textbox = gr.Textbox(label="Введите сообщение")
            send_button = gr.Button("Отправить")

            # Пример возможных state (можно менять или добавлять свои)
            pending_interrupt_state = gr.State(False)
            thread_id_state = gr.State(str(uuid.uuid4()))

            # Привязываем колбеки
            demo.load(
                fn=self.on_load,
                inputs=[chatbot, pending_interrupt_state, thread_id_state],
                outputs=[chatbot, pending_interrupt_state, thread_id_state]
            )

            send_button.click(
                fn=self.on_submit,
                inputs=[textbox, chatbot, pending_interrupt_state, thread_id_state],
                outputs=[chatbot, pending_interrupt_state, thread_id_state],
            )

            textbox.submit(
                fn=self.on_submit,
                inputs=[textbox, chatbot, pending_interrupt_state, thread_id_state],
                outputs=[chatbot, pending_interrupt_state, thread_id_state],
            )
        return demo